# 自由探索用 notebook
在這裡試新的分析；確定好用之後再寫成 `lta/analyses/` 裡的一個分析（見 README「新增一種分析」）。

In [ ]:
%matplotlib inline
import os, sys
import numpy as np
import matplotlib.pyplot as plt

TOOL = os.path.abspath(".")          # 這個 notebook 所在的工具資料夾
sys.path.insert(0, TOOL)

from lta import QDB
from lta.batch.runner import load_settings, run_job_file
from lta.catalog import newest_db
from lta.core.common import clean_sort, derivative, linearize_edges, robust_linear_fit

settings = load_settings()

In [ ]:
# 指定 .db；留空字串 = 用 data_root 底下最新的 .db
DB_PATH = r""
db = QDB(DB_PATH or newest_db(settings["data_root"]))
print(db.path)
print(db.table(db.runs()[-20:]))       # 最後 20 筆；全部用 db.table()

## 1D：讀一個 run

In [ ]:
ds = db.load(116)
c = ds.curve()                          # 一般化：c.x（施加）、c.y（量測），V→I 也適用
print(c.mode, c.x.scaled()[1], c.y.scaled()[1])
I, V = ds.xy()                          # A, V（I→V 專用的捷徑）
I, V = clean_sort(I, V)                 # 排序、同電流取平均
Im, dVdI = derivative(I, V)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(I / 1e-6, V / 1e-3); ax1.set_xlabel("I (μA)"); ax1.set_ylabel("V (mV)")
ax2.plot(Im / 1e-6, dVdI);    ax2.set_xlabel("I (μA)"); ax2.set_ylabel("dV/dI (Ω)")
fig.suptitle(ds.info.exp_name); plt.show()

## 依條件挑 run（溫度、接點自動從實驗描述解析）

In [ ]:
runs = db.select(contacts="I+_38, V+_39, V-_40, I-_41", kind="IV",
                 T_range=[1.6, 3.0], sweep="none", duplicates="latest")
for r in runs:
    print(r.run_id, r.T_K, r.exp_name)

## 2D：磁場掃描

In [ ]:
m = db.load(142).map2d()                 # m.slow / m.fast / m.z，各是 (慢軸數, 快軸數)
B, I2, V2 = m.slow.values, m.fast.values, m.z.values
plt.figure(figsize=(7, 5))
plt.pcolormesh(B / 1e-3, I2 / 1e-6, V2 / 1e-3, cmap="magma", shading="auto")
plt.colorbar(label="V (mV)"); plt.xlabel("B (mT)"); plt.ylabel("I (μA)"); plt.show()

## 從 notebook 跑設定檔（結果一樣存到 01 Figure，並更新 index.html）

In [ ]:
# run_job_file(os.path.join(TOOL, "jobs", "quad_002.toml"), only=["dVdI_vs_T_log"])

## 直接用介面上的分析功能畫圖


In [ ]:
from lta.core.registry import load_analyses, Item
from matplotlib.figure import Figure
reg = load_analyses(reload=True)            # 改了 analysis1d/ 的程式後重新執行這格即可
a = next(x for x in reg["1D"] if x.name.startswith("微分"))
fig = plt.figure(figsize=(9, 5), layout="constrained")
items = [Item(db.load(r), f"run {r}", f"C{k}") for k, r in enumerate([116, 110, 104])]
params = {p.key: p.default for p in a.params}
params["ylog"] = True
res = a.func(fig, items, params)
print(res.text)
plt.show()